# Expertise: from data entry to RDF

This notebook shows how to describe a person's areas of expertise and convert
that description into a standardised, machine-readable RDF graph.

**You only need to edit one file:** `docs/example.oold.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.oold.json
  │  URIs for each area of expertise
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
(no SHACL shape yet; structural correctness enforced by JSON Schema)
```

> **No transform step:** the expertise input is already in OO-LD format;
> the fields map directly to ontology IRIs.  There is nothing to pre-process.

---

## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # expertise/

schema = Schema(SCHEMA)

print("Python :", sys.executable)

Python : /root/semantic-dataspace/.venv/bin/python3


---
## Step 1: Describe a person's expertise

Edit `docs/example.oold.json` with your data, then run this cell to load it.

Each field is an array of knowledge-graph URIs, one per concept
(e.g. a material, a process, a measurement method) that this person
works with.  The available URIs come from your DSMS knowledge graph.

> **Tip:** you can omit any field whose array would be empty; leave it out
> entirely rather than setting it to `[]`.

In [3]:
doc = json.loads((HERE / "example.oold.json").read_text())

print(json.dumps(doc, indent=2))

{
  "type": "foaf:Person",
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/expertise/#v1.0.0",
  "materials": [
    "https://dsms.example.org/api/knowledge/mat-steel-316l",
    "https://dsms.example.org/api/knowledge/mat-alsi10mg"
  ],
  "material_modelling": [
    "https://dsms.example.org/api/knowledge/sim-dft",
    "https://dsms.example.org/api/knowledge/sim-fem"
  ],
  "measurement_devices": [
    "https://dsms.example.org/api/knowledge/dev-sem",
    "https://dsms.example.org/api/knowledge/dev-xrd"
  ],
  "production_devices": [
    "https://dsms.example.org/api/knowledge/dev-lpbf"
  ],
  "application_fields": [
    "https://dsms.example.org/api/knowledge/app-aerospace"
  ],
  "methods": [
    "https://dsms.example.org/api/knowledge/method-ebsd",
    "https://dsms.example.org/api/knowledge/method-tensile-testing"
  ]
}


---
## Step 2: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [4]:
# The expertise schema uses a pre-built OO-LD document (example.oold.json).
# schema.parse() skips the JSONata transform and goes straight to RDF.
flat = schema.parse(doc)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 12 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix schema: <https://schema.org/> .

[] a foaf:Person ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/expertise/#v1.0.0> ;
    schema:knowsAbout <https://dsms.example.org/api/knowledge/app-aerospace>,
        <https://dsms.example.org/api/knowledge/dev-lpbf>,
        <https://dsms.example.org/api/knowledge/dev-sem>,
        <https://dsms.example.org/api/knowledge/dev-xrd>,
        <https://dsms.example.org/api/knowledge/mat-alsi10mg>,
        <https://dsms.example.org/api/knowledge/mat-steel-316l>,
        <https://dsms.example.org/api/knowledge/method-ebsd>,
        <https://dsms.example.org/api/knowledge/method-tensile-testing>,
        <https://dsms.example.org/api/knowledge/sim-dft>,
        <https://dsms.example.org/api/knowledge/sim-fem> .




In [5]:
# Optional: save to file
out_ttl = HERE / "output_expertise.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_expertise.ttl


---
## Step 3: Inspect the graph

The cell below uses SPARQL to list all expertise areas stored in the graph.

You do not need to understand SPARQL to use this notebook. Just run the cell
and check that every IRI you entered in Step 1 appears in the output.

In [6]:
SPARQL_EXPERTISE = """
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX schema: <https://schema.org/>

SELECT ?expertise
WHERE {
  ?person a foaf:Person ;
          schema:knowsAbout ?expertise .
}
ORDER BY ?expertise
"""

rows = list(flat.query(SPARQL_EXPERTISE))
if rows:
    print(f"Expertise areas ({len(rows)}):")
    for r in rows:
        print(f"  {r.expertise}")
else:
    print("No expertise areas recorded.")

Expertise areas (10):
  https://dsms.example.org/api/knowledge/app-aerospace
  https://dsms.example.org/api/knowledge/dev-lpbf
  https://dsms.example.org/api/knowledge/dev-sem
  https://dsms.example.org/api/knowledge/dev-xrd
  https://dsms.example.org/api/knowledge/mat-alsi10mg
  https://dsms.example.org/api/knowledge/mat-steel-316l
  https://dsms.example.org/api/knowledge/method-ebsd
  https://dsms.example.org/api/knowledge/method-tensile-testing
  https://dsms.example.org/api/knowledge/sim-dft
  https://dsms.example.org/api/knowledge/sim-fem


---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.oold.json` with the relevant knowledge-graph URIs |
| 2 | The OO-LD document is parsed into an RDF graph |
| 3 | Expertise areas are extracted from the graph to confirm correctness |

To describe a different expert, edit `example.oold.json` and re-run all cells.

---

## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/3_schema-format.md): how to write your own schema